# Nemotron GRPO v3 — Hard Categories with vLLM Rollouts

## Architecture
- **vLLM** (from NVIDIA metric utility script) for batch rollout generation (~10 min vs ~5 hrs with HF generate)
- **HuggingFace + PEFT** for gradient-based policy optimization (log-probs + backward)

## Key innovations vs previous GRPO attempts
1. **Partial-credit rewards** — Hamming distance for bit_manipulation, char-level for cryptarithm
2. **vLLM batch generation** — generate ALL rollouts upfront in one fast batch
3. **Dual-adapter** — policy (trainable) vs reference (frozen SFT checkpoint)

## Category-specific rewards
| Category | Reward | Method |
|---|---|---|
| bit_manipulation | Partial | matching_bits / 8 (Hamming) |
| cryptarithm | Partial | positional char match / length |
| equation_numeric_guess | Partial | digit overlap + exact bonus |

## Workflow
1. Load vLLM with SFT adapter → batch-generate K rollouts for all prompts
2. Free vLLM GPU memory
3. Load HF model with dual LoRA (policy + reference)
4. GRPO training loop on pre-generated rollouts
5. Save optimized adapter for submission

## Expected runtime
- vLLM rollout generation: ~10-15 min
- HF GRPO training (80 updates): ~20-40 min
- Total: **~30-60 min** (vs 5-6 hrs without vLLM)

In [ ]:
# ============================================================
# 1. ENVIRONMENT SETUP (NVIDIA metric utility script → vLLM)
# ============================================================
import subprocess, sys, os, glob, stat, shutil
from pathlib import Path

IN_KAGGLE = os.path.exists('/kaggle')

# ── NVIDIA metric utility script (provides vLLM + PyTorch + transformers) ──
if IN_KAGGLE:
    for cmd in [
        'uv pip uninstall torch torchvision torchaudio',
        'tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp',
    ]:
        print(f'Running: {cmd}')
        subprocess.run(cmd, shell=True, check=False)
    for f in glob.glob('/tmp/triton/backends/nvidia/bin/ptxas*'):
        os.chmod(f, os.stat(f).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    sys.path.insert(0, '/tmp')

    # ── Fix: remove stale ryanholbrook nvidia_utility_script from sys.path ──
    # Its mamba_ssm .so was compiled against the old PyTorch we just uninstalled.
    # /kaggle/usr is READ-ONLY so we can't rename/delete — just remove from path.
    for p in list(sys.path):
        if 'ryanholbrook' in p:
            sys.path.remove(p)
            print(f'[fix] removed stale path: {p}')

    # ── Install correct mamba_ssm + causal_conv1d from nemotron packages dataset ──
    # These wheels match torch 2.10 + CUDA 12 from the metric utility script.
    _installed = []
    for _whl in sorted(
        glob.glob('/kaggle/input/**/mamba_ssm*.whl', recursive=True)
        + glob.glob('/kaggle/input/**/causal_conv1d*.whl', recursive=True)
    ):
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', _whl],
            check=False,
        )
        _installed.append(os.path.basename(_whl))
    for w in _installed:
        print(f'[fix] installed wheel: {w}')
    if not _installed:
        print('[warn] no mamba_ssm/causal_conv1d wheels found in /kaggle/input/')

# ── Additional packages (PEFT for LoRA training) ──
TARGET_DIR = "/kaggle/working/packages" if IN_KAGGLE else "./packages"
os.makedirs(TARGET_DIR, exist_ok=True)
if TARGET_DIR not in sys.path:
    sys.path.insert(1, TARGET_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
     "--target", TARGET_DIR, "peft"],
    check=False,
)

# Resolve .pth files
for pth in Path(TARGET_DIR).glob("*.pth"):
    with pth.open() as fp:
        p = pth.parent / fp.read().strip()
        if p.exists() and str(p) not in sys.path:
            sys.path.append(str(p))

os.environ.update({
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TOKENIZERS_PARALLELISM": "false",
    "WANDB_MODE": "offline",
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_FLAX": "1",
})

import torch, transformers
print(f"torch {torch.__version__} | transformers {transformers.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
print("[ok] vLLM available")

from peft import PeftModel
print("[ok] PEFT available")

# Verify mamba_ssm loads from the correct path (not stale ryanholbrook)
try:
    import mamba_ssm
    print(f"[ok] mamba_ssm {mamba_ssm.__version__} from {mamba_ssm.__file__}")
except ImportError as e:
    print(f"[warn] mamba_ssm import: {e}")

In [ ]:
# ============================================================
# 2. CONFIG + IMPORTS + TOKENIZER
# ============================================================
import json, time, re, math, hashlib, random, zipfile, csv, gc
import multiprocessing
from collections import Counter, defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
csv.field_size_limit(sys.maxsize)

# ── Paths ──
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

# *** YOUR 0.86 SFT ADAPTER — edit this path ***
SFT_ADAPTER_PATH = "/kaggle/input/models/manish756/nvidia-adapter/transformers/default/7"

# Dataset — JSONL files in a Kaggle dataset folder (not a zip)
DATASET_DIR_CANDIDATES = [
    "/kaggle/input/v8-fresh-dataset",
    "/kaggle/input/v8-fresh",
    "/kaggle/input/datasets/manish756/v8-fresh-dataset",
    "/kaggle/input/datasets/manish756/v8-fresh",
    "/kaggle/input/all-categorical-splits-v8-fresh",
    "/kaggle/input/all-new-nemotron-cot",
]

# train.csv for GT answer verification
TRAIN_CSV_CANDIDATES = [
    "/kaggle/input/datasets/manish756/nemotron-dataset/train.csv",
    "/kaggle/input/nvidia-puzzles/train.csv",
    "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv",
]

# ── GRPO focus categories ──
GRPO_CATEGORIES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
]

CATEGORY_WEIGHTS = {
    "bit_manipulation": 3.0,
    "cryptarithm_deduce": 2.0,
    "cryptarithm_guess": 2.0,
    "equation_numeric_guess": 1.5,
}

# ── Per-category rollout temperature ──
# bit_manipulation needs LOWER temp to stay close to greedy behavior
# (model gets 81.5% at temp=0 but 0% at temp=0.8 — too much randomness
#  destroys the precise bit-level reasoning).
# Other categories benefit from higher diversity for exploration.
CATEGORY_TEMP = {
    "bit_manipulation": 0.4,
    "cryptarithm_deduce": 0.8,
    "cryptarithm_guess": 0.8,
    "equation_numeric_guess": 0.7,
}

# ── Rollout settings (vLLM) ──
K_ROLLOUTS       = 4          # completions per prompt
ROLLOUT_TEMP     = 0.8        # default temp (overridden by CATEGORY_TEMP)
ROLLOUT_TOP_P    = 0.95
MAX_NEW_TOKENS   = 2048       # was 1024 — too short, truncated CoT before \\boxed{}

# ── GRPO hyperparameters ──
GRPO_LR          = 5e-7
KL_COEF          = 0.04
MAX_KL_PER_STEP  = 0.15
ADV_NORM         = True
MAX_GRAD_NORM    = 0.3

# ── Training budget ──
NUM_ACCEPTED_UPDATES  = 80
SAVE_EVERY            = 20

# ── Reward weights ──
EXACT_MATCH_BONUS = 1.0
FORMAT_BONUS      = 0.02
THINK_BONUS       = 0.01

# ── Outputs ──
OUTPUT_DIR = "/kaggle/working/grpo_v3_adapter"
CKPT_DIR   = "/kaggle/working/grpo_v3_checkpoints"
BEST_DIR   = "/kaggle/working/grpo_v3_best"
for d in [OUTPUT_DIR, CKPT_DIR, BEST_DIR]:
    os.makedirs(d, exist_ok=True)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# ── Tokenizer (shared between vLLM prompt formatting and HF training) ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def apply_chat(user_text: str) -> str:
    msgs = [{'role': 'user', 'content': user_text}]
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
            enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)

print("=" * 60)
print("GRPO v3 — Hard Categories with vLLM Rollouts")
print("=" * 60)
for k in ["K_ROLLOUTS", "MAX_NEW_TOKENS",
          "GRPO_LR", "KL_COEF", "NUM_ACCEPTED_UPDATES"]:
    print(f"  {k:24s}: {globals()[k]}")
print("  Per-category temperatures:")
for cat, t in CATEGORY_TEMP.items():
    print(f"    {cat:28s}: {t}")

In [ ]:
# ============================================================
# 4. PARTIAL-CREDIT REWARD FUNCTIONS
# ============================================================
# The key innovation: category-aware rewards that give gradient
# signal even when the answer isn't perfectly correct.

def extract_boxed(text: str) -> str | None:
    """Extract last \\boxed{...} content. Handles } inside answers."""
    if not text:
        return None
    matches = list(re.finditer(r'\\boxed\{', text))
    if not matches:
        return None
    start = matches[-1].end()
    end = text.rfind('}')
    if end > start:
        return text[start:end].strip()
    return None


def reward_bit_manipulation(pred: str, gt: str) -> float:
    """Partial credit: fraction of matching bits.

    bit_manipulation answers are binary strings of VARYING length
    (not always 8 bits). Handles whitespace and common formatting issues.
    """
    if pred is None:
        return 0.0
    # Clean up — model may output spaces, newlines, quotes around answer
    pred = pred.strip().strip('"').strip("'").replace(' ', '')
    gt = gt.strip()
    # Both must be binary strings
    if not gt or not all(c in '01' for c in gt):
        return 0.0
    if not pred or not all(c in '01' for c in pred):
        return 0.0
    # Exact match
    if pred == gt:
        return 1.0
    # Same length — Hamming distance (fraction of matching bits)
    if len(pred) == len(gt):
        matching = sum(1 for a, b in zip(pred, gt) if a == b)
        return matching / len(gt)
    # Wrong length — no partial credit (different bit count = wrong approach)
    return 0.0


def reward_cryptarithm(pred: str, gt: str) -> float:
    """Partial credit: character-level matching.

    Cryptarithm answers are symbol strings like '@&' or '[](}'.
    If the model outputs the right length with some matching chars,
    it gets partial credit proportional to the overlap.
    """
    if pred is None:
        return 0.0
    if pred == gt:
        return 1.0
    # Length match bonus
    if len(pred) != len(gt):
        # Wrong length — very small credit if any chars match
        common = sum(1 for c in pred if c in gt)
        return 0.05 * min(common / max(len(gt), 1), 1.0)
    # Right length — count positional matches
    matching = sum(1 for a, b in zip(pred, gt) if a == b)
    return matching / len(gt)


def reward_numeric_guess(pred: str, gt: str) -> float:
    """Partial credit for numeric answers.

    equation_numeric_guess answers are like '731', '-62', '*53'.
    Exact match = 1.0, otherwise partial based on character overlap.
    """
    if pred is None:
        return 0.0
    if pred.strip() == gt.strip():
        return 1.0
    # Partial: character-level matching (positional)
    if len(pred) == len(gt):
        matching = sum(1 for a, b in zip(pred, gt) if a == b)
        return 0.5 * matching / max(len(gt), 1)
    return 0.0


def reward_fn(generated_text: str, gt_answer: str, category: str) -> float:
    """Category-aware reward with partial credit + format bonuses."""
    pred = extract_boxed(generated_text)
    reward = 0.0

    # Format bonuses (tiny — prevent format collapse)
    if pred is not None:
        reward += FORMAT_BONUS
    if '</think>' in (generated_text or ''):
        reward += THINK_BONUS

    # Category-specific partial credit
    if category == 'bit_manipulation':
        reward += reward_bit_manipulation(pred, gt_answer)
    elif category in ('cryptarithm_deduce', 'cryptarithm_guess'):
        reward += reward_cryptarithm(pred, gt_answer)
    elif category == 'equation_numeric_guess':
        reward += reward_numeric_guess(pred, gt_answer)
    else:
        # Binary reward for other categories
        if pred is not None and pred.strip() == gt_answer.strip():
            reward += EXACT_MATCH_BONUS

    return reward


# Test the reward functions
print("Reward function tests:")
print(f"  bit_manip  7/8 right:   {reward_bit_manipulation('10110101', '10110100'):.3f}")
print(f"  bit_manip  8/8 right:   {reward_bit_manipulation('10110101', '10110101'):.3f}")
print(f"  bit_manip  wrong len:   {reward_bit_manipulation('1011', '10110101'):.3f}")
print(f"  bit_manip  16-bit:      {reward_bit_manipulation('1011010110110100', '1011010110110101'):.3f}")
print(f"  bit_manip  with spaces: {reward_bit_manipulation(' 10110101 ', '10110101'):.3f}")
print(f"  crypto     exact:       {reward_cryptarithm('@&', '@&'):.3f}")
print(f"  crypto     1/2 right:   {reward_cryptarithm('@!', '@&'):.3f}")
print(f"  crypto     wrong len:   {reward_cryptarithm('@&!', '@&'):.3f}")
print(f"  numeric    exact:       {reward_numeric_guess('731', '731'):.3f}")
print(f"  numeric    partial:     {reward_numeric_guess('732', '731'):.3f}")
print(f"  numeric    wrong:       {reward_numeric_guess('999', '731'):.3f}")

In [ ]:
# ============================================================
# 4. LOAD PROMPTS FROM DATASET + FORMAT FOR vLLM
# ============================================================

# Find dataset directory containing JSONL files
DATA_DIR = None
for cand in DATASET_DIR_CANDIDATES:
    if os.path.isdir(cand):
        # Check it actually has at least one of our target JSONL files
        if any(os.path.isfile(os.path.join(cand, f)) for f in GRPO_CATEGORIES):
            DATA_DIR = cand
            break

# Fallback: recursive search under /kaggle/input
if DATA_DIR is None:
    for fp in glob.glob("/kaggle/input/**/train_cot_bit_manipulation.jsonl", recursive=True):
        DATA_DIR = os.path.dirname(fp)
        break

# Also find train.csv for GT verification
TRAIN_CSV = None
for cand in TRAIN_CSV_CANDIDATES:
    if os.path.exists(cand):
        TRAIN_CSV = cand
        break
if TRAIN_CSV is None:
    for fp in glob.glob("/kaggle/input/**/train.csv", recursive=True):
        TRAIN_CSV = fp
        break

# Load GT answers from train.csv
gt_by_id = {}
if TRAIN_CSV:
    with open(TRAIN_CSV) as f:
        for row in csv.DictReader(f):
            gt_by_id[row['id']] = row['answer']
    print(f"[ok] train.csv: {len(gt_by_id)} GT answers")

# Load prompts from GRPO focus categories
records = []

def load_jsonl_records(fname, source):
    """Load records from a JSONL string, return list of dicts."""
    cat = fname.replace('train_cot_', '').replace('.jsonl', '')
    loaded = []
    for line in source.strip().split('\n'):
        if not line.strip():
            continue
        rec = json.loads(line)
        msgs = [m for m in rec.get('messages', []) if m.get('role') != 'system']
        if len(msgs) < 2:
            continue
        user = msgs[0]['content']
        asst = msgs[-1]['content']
        rid = rec.get('id', hashlib.md5(user.encode()).hexdigest()[:8])

        # Use GT from train.csv if available, otherwise extract from CoT
        gt_answer = gt_by_id.get(rid)
        if gt_answer is None:
            gt_answer = extract_boxed(asst)
        if gt_answer is None:
            continue

        loaded.append({
            'id': rid,
            'category': cat,
            'user': user,
            'gt_answer': gt_answer,
            'weight': CATEGORY_WEIGHTS.get(cat, 1.0),
        })
    return loaded


if DATA_DIR is None:
    raise FileNotFoundError(
        "No dataset directory found! Tried:\n"
        + "\n".join(f"  - {c}" for c in DATASET_DIR_CANDIDATES)
        + "\nAlso searched /kaggle/input/ recursively."
    )

print(f"Loading from dir: {DATA_DIR}")
for fname in GRPO_CATEGORIES:
    fp = os.path.join(DATA_DIR, fname)
    if os.path.exists(fp):
        with open(fp) as f:
            data = f.read()
        recs = load_jsonl_records(fname, data)
        records.extend(recs)
        print(f"  {len(recs):>5} from {fname}")
    else:
        print(f"  [skip] {fname} not found in {DATA_DIR}")

# Dedupe by ID
seen = set()
deduped = []
for r in records:
    if r['id'] not in seen:
        seen.add(r['id'])
        deduped.append(r)
records = deduped

# Format prompts with chat template (used by both vLLM and HF)
for r in records:
    r['prompt_text'] = apply_chat(r['user'])

print(f"\nTotal unique prompts: {len(records)}")
print("Per category:")
for cat, n in Counter(r['category'] for r in records).most_common():
    w = CATEGORY_WEIGHTS.get(cat, 1.0)
    print(f"  {cat:30s} {n:>5}  weight={w}")

In [ ]:
# ============================================================
# 6. vLLM BATCH ROLLOUT GENERATION
# ============================================================
# Generate ALL rollouts upfront with vLLM (fast), then free GPU
# for HF training phase. This replaces the slow HF model.generate().

def cache_model(path, exts=('.bin', '.pt', '.safetensors'),
                num_workers=None, chunk_mb=256):
    """Pre-read model files into OS page cache for faster loading."""
    path = Path(path)
    files = [p for p in path.rglob('*')
             if p.is_file() and any(str(p).endswith(e) for e in exts)]
    if not files:
        return 0
    num_workers = min(multiprocessing.cpu_count(), 8) \
                  if num_workers is None else num_workers
    t0 = time.time()
    total = 0
    def _read(f):
        n = 0
        with open(f, 'rb') as fh:
            while True:
                d = fh.read(chunk_mb * 1024 * 1024)
                if not d:
                    break
                n += len(d)
        return n
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        for n in pool.map(_read, files):
            total += n
    print(f"[cache] {total / 1e9:.1f} GB in {time.time() - t0:.0f}s")
    return total


# ── Find SFT adapter ──
adapter_path = SFT_ADAPTER_PATH
if not os.path.exists(os.path.join(adapter_path, 'adapter_config.json')):
    for cfg in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True):
        d = os.path.dirname(cfg)
        if os.path.exists(os.path.join(d, 'adapter_model.safetensors')):
            adapter_path = d
            break

with open(os.path.join(adapter_path, 'adapter_config.json')) as f:
    acfg = json.load(f)
LORA_RANK = acfg.get('r', 32)
print(f"SFT adapter: r={LORA_RANK} alpha={acfg.get('lora_alpha')}")
print(f"  path: {adapter_path}")

# ── Cache model files ──
cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

# ── Load vLLM engine ──
print("\nLoading vLLM engine...")
t0 = time.time()
llm = LLM(
    model=str(MODEL_PATH),
    enable_lora=True,
    max_lora_rank=LORA_RANK,
    tensor_parallel_size=1,
    max_num_seqs=64,
    gpu_memory_utilization=0.85,
    dtype='auto',
    max_model_len=8192,
    trust_remote_code=True,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)
print(f"vLLM ready in {time.time() - t0:.0f}s")

# ── Tokenizer consistency check ──
vllm_tok = llm.get_tokenizer()
_test_str = records[0]['prompt_text']
_hf_ids   = tokenizer(_test_str, add_special_tokens=False).input_ids
_vllm_ids = vllm_tok(_test_str, add_special_tokens=False).input_ids
if _hf_ids == _vllm_ids:
    print(f"[ok] Tokenizer consistent ({len(_hf_ids)} tokens)")
else:
    print(f"[warn] Tokenizer mismatch: HF={len(_hf_ids)} vLLM={len(_vllm_ids)}")

# ── Pre-tokenize all prompts + build per-prompt sampling params ──
# Each category gets its own temperature from CATEGORY_TEMP
all_token_ids = []
prompt_indices = []
all_sampling_params = []

for i, r in enumerate(records):
    ids = tokenizer(r['prompt_text'], add_special_tokens=False).input_ids
    temp = CATEGORY_TEMP.get(r['category'], ROLLOUT_TEMP)
    sp = SamplingParams(
        temperature=temp,
        top_p=ROLLOUT_TOP_P,
        max_tokens=MAX_NEW_TOKENS,
    )
    for _ in range(K_ROLLOUTS):
        all_token_ids.append(ids)
        prompt_indices.append(i)
        all_sampling_params.append(sp)

lora_req = LoRARequest('sft', 1, adapter_path)

# Show per-category generation plan
_cat_counts = Counter(r['category'] for r in records)
print(f"\nGenerating {len(all_token_ids)} completions "
      f"({len(records)} prompts x {K_ROLLOUTS} rollouts):")
for cat, cnt in _cat_counts.most_common():
    t = CATEGORY_TEMP.get(cat, ROLLOUT_TEMP)
    print(f"  {cat:30s} {cnt:>5} prompts  temp={t}")

t0 = time.time()
outputs = llm.generate(
    prompt_token_ids=all_token_ids,
    sampling_params=all_sampling_params,
    lora_request=lora_req,
)
gen_time = time.time() - t0

# ── Process outputs into rollout groups ──
prompt_rollouts = defaultdict(list)
total_tokens = 0

for out_idx, output in enumerate(outputs):
    i = prompt_indices[out_idx]
    item = records[i]
    completion = output.outputs[0]
    text = completion.text
    token_ids = list(completion.token_ids)
    total_tokens += len(token_ids)

    reward = reward_fn(text, item['gt_answer'], item['category'])
    exact = 1.0 if extract_boxed(text) == item['gt_answer'] else 0.0

    prompt_rollouts[i].append({
        'text': text,
        'token_ids': token_ids,
        'reward': reward,
        'exact': exact,
    })

print(f"\nGenerated {total_tokens:,} tokens in {gen_time:.0f}s "
      f"({total_tokens / max(gen_time, 1):.0f} tok/s)")

# ── Identify prompts with reward variance (will produce gradient signal) ──
with_variance = []
for i, rolls in prompt_rollouts.items():
    rewards = [r['reward'] for r in rolls]
    if max(rewards) - min(rewards) > 1e-6:
        with_variance.append(i)

print(f"\nPrompts with reward variance: {len(with_variance)}/{len(prompt_rollouts)}")

# ── Per-category breakdown ──
print("\nPer-category rollout stats:")
for cat in sorted(set(r['category'] for r in records)):
    cat_idx = [i for i, r in enumerate(records) if r['category'] == cat]
    cat_r = [r['reward'] for i in cat_idx for r in prompt_rollouts[i]]
    cat_e = [r['exact'] for i in cat_idx for r in prompt_rollouts[i]]
    cat_v = sum(1 for i in cat_idx if i in set(with_variance))
    if cat_r:
        print(f"  {cat:30s} R={sum(cat_r)/len(cat_r):.3f} "
              f"exact={sum(cat_e)/len(cat_e):.3f} "
              f"var={cat_v}/{len(cat_idx)}")

# ── Free vLLM GPU memory ──
print("\nFreeing vLLM memory...")
del llm, outputs, all_token_ids, all_sampling_params, vllm_tok
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ============================================================
# 7. LOAD HF MODEL + DUAL ADAPTER (for gradient-based training)
# ============================================================
# vLLM is gone. Now load HF model for log-prob computation + backward pass.

print("Loading base model in bf16...")
t0 = time.time()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={'': 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation='eager',
)
base_model.config.use_cache = False
try:
    base_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={'use_reentrant': False})
except Exception as e:
    print(f"[warn] grad ckpt: {e}")
print(f"Base loaded in {time.time()-t0:.0f}s | "
      f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# ── Load policy adapter (trainable) ──
print("Loading policy adapter (trainable)...")
model = PeftModel.from_pretrained(
    base_model, adapter_path,
    adapter_name='default', is_trainable=True,
)

# ── Load reference adapter (frozen) ──
print("Loading reference adapter (frozen)...")
model.load_adapter(adapter_path, adapter_name='ref', is_trainable=False)

# ── Freeze everything except policy LoRA ──
for name, p in model.named_parameters():
    p.requires_grad = ('lora_' in name and '.default.' in name)
    if p.requires_grad:
        p.data = p.data.float()  # fp32 for LoRA stability

model.set_adapter('default')
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable params: {sum(p.numel() for p in trainable)/1e6:.1f}M")

# Sanity check
for name, p in model.named_parameters():
    if p.requires_grad and 'lora_B' in name:
        ma = p.detach().abs().mean().item()
        print(f"Sanity: mean_abs={ma:.6f}")
        assert ma > 1e-8, "Adapter is zero — load failed!"
        break

print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


# ── Helper: compute mean log-probs ──
def mean_log_probs(input_ids, prompt_len, completion_mask, adapter_name, grad):
    """Mean log-prob of completion tokens under given adapter."""
    model.set_adapter(adapter_name)
    with torch.set_grad_enabled(grad):
        logits = model(input_ids=input_ids, use_cache=False).logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        logp = F.log_softmax(shift_logits.float(), dim=-1)
        tok_logp = logp.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)

        mask = torch.zeros_like(tok_logp, dtype=torch.float32)
        c_len = completion_mask.shape[1]
        mask[:, prompt_len - 1:prompt_len - 1 + c_len] = completion_mask

        denom = mask.sum(dim=-1).clamp(min=1.0)
        return (tok_logp * mask).sum(dim=-1) / denom


def compute_advantages(rewards: torch.Tensor) -> torch.Tensor:
    adv = rewards - rewards.mean()
    if ADV_NORM and rewards.std() > 1e-6:
        adv = adv / (rewards.std() + 1e-6)
    return adv


print("[ok] HF model + training primitives ready")

In [ ]:
# ============================================================
# 8. GRPO TRAINING LOOP (on pre-generated vLLM rollouts)
# ============================================================
# No more model.generate() — we use the rollouts from Cell 6.
# Each step: build full_ids → policy log-probs (grad) → ref log-probs
# (no grad) → GRPO loss → backward → update.

optimizer = torch.optim.AdamW(
    trainable, lr=GRPO_LR,
    betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
)

metrics = defaultdict(list)
recent_reward = deque(maxlen=40)
best_recent = -1.0
accepted = 0
skipped_kl = 0
t_start = time.time()

cat_rewards_log = defaultdict(list)
cat_exact_log = defaultdict(list)


def save_adapter(path: str):
    os.makedirs(path, exist_ok=True)
    model.set_adapter('default')
    model.save_pretrained(path, selected_adapters=['default'])
    cfg_path = os.path.join(path, 'adapter_config.json')
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            cfg = json.load(f)
        cfg['base_model_name_or_path'] = 'metric/nemotron-3-nano-30b-a3b-bf16'
        cfg['lora_dropout'] = 0.0
        with open(cfg_path, 'w') as f:
            json.dump(cfg, f, indent=2)


# ── Build training order (shuffle, each prompt used at most once) ──
train_order = list(with_variance)
random.shuffle(train_order)

print('=' * 70)
print(f'GRPO v3 — {len(with_variance)} prompts with reward variance')
print(f'Target: {NUM_ACCEPTED_UPDATES} accepted updates')
print('=' * 70)

attempted = 0
for prompt_idx in train_order:
    if accepted >= NUM_ACCEPTED_UPDATES:
        break

    attempted += 1
    item = records[prompt_idx]
    rollouts = prompt_rollouts[prompt_idx]
    t_step = time.time()

    # ── Rewards + advantages ──
    rewards_list = [r['reward'] for r in rollouts]
    exact_list = [r['exact'] for r in rollouts]
    rewards = torch.tensor(rewards_list, device='cuda', dtype=torch.float32)

    if rewards.std() < 1e-6:
        continue  # double-check variance

    cat_rewards_log[item['category']].append(float(rewards.mean()))
    cat_exact_log[item['category']].append(
        float(sum(exact_list) / len(exact_list)))

    adv = compute_advantages(rewards)

    # ── Build full_ids from prompt + vLLM completion token IDs ──
    prompt_ids = tokenizer(
        item['prompt_text'], return_tensors='pt',
        add_special_tokens=False,
    ).input_ids.to('cuda')
    prompt_len = prompt_ids.shape[1]

    comp_lengths = [len(r['token_ids']) for r in rollouts]
    max_comp_len = max(comp_lengths)
    if max_comp_len == 0:
        continue

    padded_comps = [
        r['token_ids']
        + [tokenizer.eos_token_id] * (max_comp_len - len(r['token_ids']))
        for r in rollouts
    ]
    comp_tensor = torch.tensor(padded_comps, device='cuda', dtype=torch.long)
    full_ids = torch.cat(
        [prompt_ids.repeat(K_ROLLOUTS, 1), comp_tensor], dim=1)

    comp_mask = torch.zeros(K_ROLLOUTS, max_comp_len, device='cuda')
    for j, clen in enumerate(comp_lengths):
        comp_mask[j, :clen] = 1.0

    # ── Policy log-probs (with grad) ──
    model.train()
    optimizer.zero_grad(set_to_none=True)
    log_pi = mean_log_probs(
        full_ids, prompt_len, comp_mask, 'default', grad=True)

    # ── Reference log-probs (no grad) ──
    with torch.no_grad():
        log_ref = mean_log_probs(
            full_ids, prompt_len, comp_mask, 'ref', grad=False)
    model.set_adapter('default')

    # ── GRPO loss ──
    log_delta = (log_pi - log_ref.detach()).clamp(-2.0, 2.0)
    sampled_kl = log_delta.mean()
    kl_penalty = (log_delta ** 2).mean()
    pg_loss = -(adv.detach() * log_pi).mean()
    loss = pg_loss + KL_COEF * kl_penalty

    # NaN guard
    if not torch.isfinite(loss):
        print(f"[HALT] non-finite loss at attempt {attempted}")
        break

    # KL spike guard
    if abs(float(sampled_kl.item())) > MAX_KL_PER_STEP:
        skipped_kl += 1
        continue

    # ── Backward + clip + step ──
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
    optimizer.step()
    accepted += 1

    # ── Logging ──
    recent_reward.append(float(rewards.mean()))
    recent_avg = sum(recent_reward) / len(recent_reward)

    metrics['attempt'].append(attempted)
    metrics['update'].append(accepted)
    metrics['loss'].append(float(loss.item()))
    metrics['reward'].append(float(rewards.mean()))
    metrics['exact'].append(float(sum(exact_list) / len(exact_list)))
    metrics['kl'].append(float(sampled_kl.item()))
    metrics['grad_norm'].append(float(grad_norm))
    metrics['category'].append(item['category'])

    print(f"  upd {accepted:03d}/{attempted:04d} "
          f"[{item['category'][:20]:<20}] "
          f"R={rewards.mean():.3f} exact={sum(exact_list):.0f}/{K_ROLLOUTS} "
          f"loss={loss.item():+.4f} KL={sampled_kl.item():+.4f} "
          f"|g|={float(grad_norm):.2f} recent={recent_avg:.3f} "
          f"{time.time()-t_step:.0f}s")

    # ── Checkpointing ──
    if accepted % SAVE_EVERY == 0:
        ckpt = os.path.join(CKPT_DIR, f'update_{accepted:04d}')
        save_adapter(ckpt)
        print(f"  [ckpt] {ckpt}")

    # ── Best model tracking ──
    if len(recent_reward) >= 15 and recent_avg > best_recent:
        best_recent = recent_avg
        save_adapter(BEST_DIR)
        print(f"  [best] recent_avg={best_recent:.4f}")

# ── Final save ──
elapsed = (time.time() - t_start) / 60
print(f"\n{'=' * 70}")
print(f"Training complete: {accepted} updates, {attempted} attempts, "
      f"{elapsed:.0f} min")
print(f"Skipped: {skipped_kl} kl-spike")
print(f"{'=' * 70}")
save_adapter(OUTPUT_DIR)
print(f"Final adapter: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# 9. PER-CATEGORY ANALYSIS
# ============================================================
print('\n' + '=' * 70)
print('PER-CATEGORY GRPO RESULTS')
print('=' * 70)

for cat in sorted(cat_rewards_log.keys()):
    rews = cat_rewards_log[cat]
    exacts = cat_exact_log[cat]
    n_attempts = len(rews)
    avg_rew = sum(rews) / len(rews) if rews else 0
    avg_exact = sum(exacts) / len(exacts) if exacts else 0
    n_with_var = sum(1 for r in rews if r > 0 and r < 1.0)

    print(f'\n  {cat}:')
    print(f'    Attempts: {n_attempts}')
    print(f'    Avg reward: {avg_rew:.4f}')
    print(f'    Avg exact match: {avg_exact:.4f}')
    print(f'    Reward variance instances: ~{n_with_var}')

# Category distribution in accepted updates
if metrics['category']:
    print('\n  Accepted updates by category:')
    for cat, n in Counter(metrics['category']).most_common():
        print(f'    {cat:30s} {n:4d}')

# Reward trajectory
print('\n  Reward trajectory (10-step windows):')
if len(metrics['reward']) >= 10:
    for i in range(0, len(metrics['reward']), 10):
        chunk = metrics['reward'][i:i+10]
        avg = sum(chunk) / len(chunk)
        bar = '█' * int(avg * 40)
        print(f'    updates {i+1:3d}-{min(i+10, len(metrics["reward"])):3d}: '
              f'R={avg:.3f} {bar}')

In [ ]:
# ============================================================
# 10. SAVE + ZIP FOR SUBMISSION
# ============================================================

# Save metrics
metrics_path = '/kaggle/working/grpo_v3_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump({k: list(v) for k, v in metrics.items()}, f, indent=2)
print(f'[ok] metrics: {metrics_path}')


def zip_dir(src_dir, zip_path):
    if os.path.exists(zip_path):
        os.remove(zip_path)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fname in sorted(os.listdir(src_dir)):
            fpath = os.path.join(src_dir, fname)
            if os.path.isfile(fpath):
                zf.write(fpath, arcname=fname)
    mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'[ok] {zip_path} ({mb:.1f} MB)')


# Zip final adapter
zip_dir(OUTPUT_DIR, '/kaggle/working/grpo_v3_adapter.zip')

# Zip best adapter if it exists
if os.path.exists(os.path.join(BEST_DIR, 'adapter_config.json')):
    zip_dir(BEST_DIR, '/kaggle/working/grpo_v3_best.zip')
else:
    print('[info] no best adapter saved')

print('\n' + '=' * 70)
print('SUBMISSION INSTRUCTIONS')
print('=' * 70)
print('1. Download grpo_v3_best.zip (if exists) or grpo_v3_adapter.zip')
print('2. Upload as Kaggle dataset')
print('3. Submit via competition notebook with the new adapter path')
print('4. Compare to 0.86 baseline')
print()
print('Key metrics to watch:')
print('  - bit_manipulation: should improve from 81.5% toward 85%+')
print('  - cryptarithm: any improvement above 0-1% is a win')
print('  - equation_numeric_guess: should improve from 6.6% toward 10%+')
print('  - Other categories should NOT regress (KL constraint protects them)')